# (IIP314W) Optimización Aplicada a Negocios
## Ayudantía 9: Repaso Integral del Curso

---

**Profesor:** Ing. Rodrigo Trigo Vilches  
**Ayudante:** Lic. Vicente Ramírez Almonacid  
**Fecha:** 6 de Mayo, 2026  
**Universidad del Desarrollo**

---

Esta es la **ayudantía final** del curso IIP314W. Su objetivo no es introducir contenido nuevo, sino integrar todos los bloques temáticos en problemas de mayor complejidad y mostrar explícitamente las conexiones entre ellos. La sesión se organiza en:

- **Sección 0:** Resumen compacto del curso (tarjeta de referencia rápida).
- **Ejercicio 1:** Problema MIP complejo con costos fijos, variables de activación y análisis post-óptimo.
- **Ejercicio 2:** Pipeline LP completo — Simplex Tableau → Dual → Precios Sombra → Análisis de Sensibilidad → Verificación computacional.
- **Ejercicio 3:** Puente entre Bloque 1 y Bloque 4: Dualidad, Holguras Complementarias y KKT como dos caras de la misma moneda.

## Tabla de Contenidos

| # | Sección | Contenido |
|:---:|:---|:---|
| 0 | [Resumen del Curso: Hoja de Ruta](#sec0) | Bloques 1–4, fórmulas clave, herramientas |
| 1 | [Ejercicio 1 — Modelamiento Complejo (MIP)](#ex1) | FrioChile S.A.: cadena de frío farmacéutica |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex1-a) | Formulación matemática |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex1-b) | Implementación Gurobipy |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex1-c) | Análisis e interpretación |
| 2 | [Ejercicio 2 — LP Integral](#ex2) | Viñedos del Valle S.A.: Simplex → Dual → Sensibilidad |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex2-a) | Forma estándar, tableau inicial |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex2-b) | Iteraciones Simplex |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex2-c) | Formulación del dual, precios sombra |
| | &nbsp;&nbsp;&nbsp;[Parte (d)](#ex2-d) | Sensibilidad en $b_1$ |
| | &nbsp;&nbsp;&nbsp;[Parte (e)](#ex2-e) | Sensibilidad en $c_1$ |
| | &nbsp;&nbsp;&nbsp;[Parte (f)](#ex2-f) | Verificación scipy + Gurobipy |
| 3 | [Ejercicio 3 — Dualidad, Holguras Complementarias y KKT](#ex3) | Operadora Turística Costera: KKT ↔ Dualidad |
| | &nbsp;&nbsp;&nbsp;[Parte (a)](#ex3-a) | Recuperar $x^*$ desde $y^*$ |
| | &nbsp;&nbsp;&nbsp;[Parte (b)](#ex3-b) | Verificación KKT |
| | &nbsp;&nbsp;&nbsp;[Parte (c)](#ex3-c) | Sensibilidad en $b_2$ |
| | &nbsp;&nbsp;&nbsp;[Parte (d)](#ex3-d) | Verificación scipy |
| — | [Resumen de la Ayudantía](#resumen) | Conexiones entre bloques, tabla de herramientas |

<a id="sec0"></a>

---

## Sección 0 — Resumen del Curso: Hoja de Ruta

### Bloque 1 — Optimización No Lineal Clásica (Ayudantías 1–3)

| Concepto | Fórmula / Idea clave | Herramienta |
|:---|:---|:---|
| **Puntos críticos** | $\nabla f(x^*) = 0$ | Cálculo manual / `scipy.optimize.minimize` |
| **Clasificación Hessiana** | $H \succ 0$: mínimo local; $H \prec 0$: máximo; indef.: punto silla | `numpy.linalg.eig` |
| **Lagrange** (igualdad) | $\nabla f = \lambda \nabla g$, $g(x)=0$ | Eliminación manual |
| **KKT** (desigualdad) | Estacionaridad: $\nabla f - \sum \mu_i \nabla g_i = 0$; HC: $\mu_i g_i(x^*)=0$; Factibilidad primal: $g_i(x^*)\leq 0$; Factibilidad dual: $\mu_i \geq 0$ | Verificación manual |
| **Condición de Slater** | $\exists\, x$ tal que $g_i(x)<0\;\forall i$ → KKT son necesarias **y** suficientes | Verificación |
| **Descenso de gradiente** | $x_{k+1} = x_k - \alpha_k \nabla f(x_k)$ | `scipy.optimize.minimize` (método `'Nelder-Mead'`, `'SLSQP'`) |

> **Conexión con Bloque 4:** Las condiciones KKT ($\mu_i g_i(x^*)=0$) son exactamente las **condiciones de holgura complementaria** de la teoría dual en PL.

### Bloque 2 — Modelamiento LP y MIP (Ayudantías 4–5)

| Tipo de modelo | Ejemplo de contexto | Variable clave | Restricción típica |
|:---|:---|:---|:---|
| **LP de producción** | Maximizar margen con recursos limitados | $x_j \geq 0$ (continua) | $\sum a_{ij} x_j \leq b_i$ |
| **LP de transporte** | Flujo mínimo de plantas a ciudades | $x_{ij} \geq 0$ | Oferta, demanda |
| **MIP — localización** | Abrir/cerrar plantas, costos fijos | $y_i \in \{0,1\}$ | $\sum_j x_{ij} \leq u_i y_i$ (Big-M) |
| **MIP — lote mínimo** | Producir al menos $L$ unidades si se activa | $y_j \in \{0,1\}$ | $x_j \geq L\, y_j$ |

**Técnica Big-M:** Para ligar una variable continua $x$ a una binaria $y$:
$$x \leq M \cdot y \quad (\text{si } y=0 \Rightarrow x=0) \qquad x \geq L \cdot y \quad (\text{si } y=1 \Rightarrow x\geq L)$$

Herramientas: `scipy.optimize.linprog` (LP puro), `gurobipy` (MIP y LP de mayor escala).

---

### Bloque 3 — Algoritmo Simplex (Ayudantías 6–7)

| Concepto | Forma Tableau | Forma Matricial |
|:---|:---|:---|
| **Forma estándar** | Agregar $s_i$ (holgura, $\leq$) o $-e_i+a_i$ (excedente+artificial, $\geq$) | $[A\mid I]\begin{bmatrix}x\\s\end{bmatrix}=b$ |
| **SBF / Base** | $m$ variables básicas, resto $=0$ | $x_B = B^{-1}b\geq 0$ |
| **Costos reducidos** | Fila CR del tableau | $\bar{c}_j = c_j - c_B^\top B^{-1} a_j$ |
| **Entra** | $\min\{\bar{c}_j\}$ (más negativo) | Mismo criterio |
| **Sale** | $\min\{b_i/a_{ik}\mid a_{ik}>0\}$ | Mismo criterio |
| **Gran M** | Penalizar $a_i$ con $-M$ (max) o $+M$ (min) | Misma lógica en $c_B$ |
| **Simplex Dual** | Sale el RHS más negativo; entra $\max(|\bar{c}_j/a_{rj}|)$ para $a_{rj}<0$ | Preserva dual-factibilidad |

---

### Bloque 4 — Dualidad y Sensibilidad (Ayudantía 8)

| Concepto | Fórmula clave | Interpretación |
|:---|:---|:---|
| **Par Primal–Dual (MAX/MIN)** | Primal MAX $c^\top x$, $Ax\leq b$ $\Leftrightarrow$ Dual MIN $b^\top y$, $A^\top y\geq c$ | Variables duales = precios sombra |
| **Dualidad débil** | $b^\top y \leq c^\top x$ para cualquier par factible | El dual da cotas al primal |
| **Dualidad fuerte** | $z^* = w^*$ en el óptimo | Verificación de optimalidad |
| **Holgura complementaria** | $y_i^*(a_i^\top x^* - b_i)=0$ y $x_j^*(c_j - a_j^\top y^*)=0$ | Recuperar primal desde dual |
| **Precio sombra** | $y_i^* = \partial z^*/\partial b_i$ | Valor marginal de relajar restricción $i$ |
| **Sensibilidad en $b$** | $\bar{b} + \Delta d_i \geq 0$ donde $d_i=$ col $i$ de $B^{-1}$ | Rango donde la base no cambia |
| **Sensibilidad en $c$** | $\bar{c}_j^{\text{nuevo}}\geq 0\;\forall j\notin B$ | Rango donde la optimalidad no cambia |

**Conexión global:** KKT (Bloque 1) $\equiv$ Holguras Complementarias (Bloque 4). Los multiplicadores de Lagrange $\mu_i$ son exactamente las variables duales $y_i^*$.

<a id="ex1"></a>

---

## Ejercicio 1 — Modelamiento Complejo (MIP)

### Contexto de Negocio: Red de Distribución Refrigerada FrioChile S.A.

**FrioChile S.A.** opera una red de cadena de frío para distribuir productos farmacéuticos en dos regiones del país: Zona Norte (N) y Zona Sur (S). La empresa evalúa qué centros de distribución refrigerados (CDR) abrir durante los próximos **dos períodos** (período 1 = invierno, período 2 = verano), considerando que la demanda varía estacionalmente.

Existen **tres ubicaciones candidatas** para instalar CDRs: $k \in \{1, 2, 3\}$. Cada CDR tiene un **costo fijo de apertura** $f_k$ (en millones de pesos) que se paga solo si se decide abrir ese CDR. Una vez abierto, el CDR opera en ambos períodos. Además, cada CDR tiene una **capacidad máxima de despacho por período** $u_k$ (en toneladas).

La empresa despacha desde los CDRs abiertos hacia dos zonas $z \in \{N, S\}$ en cada período $t \in \{1, 2\}$. El **costo variable de despacho** por tonelada desde el CDR $k$ hacia la zona $z$ en el período $t$ es $c_{kzt}$. Cada zona tiene una **demanda mínima** $d_{zt}$ que debe ser satisfecha completamente.

**Restricciones adicionales:**
1. Por regulación sanitaria, **al menos 2 CDRs deben estar abiertos**.
2. Si el CDR 1 está abierto, entonces el CDR 3 **también debe estar abierto**.
3. El presupuesto total para costos fijos no puede superar **\$28 millones**.

**Parámetros numéricos:**

| CDR | Costo Fijo $f_k$ (MM\$) | Capacidad $u_k$ (ton/período) |
|:---:|:---:|:---:|
| 1 | 10 | 50 |
| 2 | 12 | 60 |
| 3 | 8 | 40 |

Costos variables $c_{kzt}$ (\$/ton):

| CDR \ Zona-Período | Zona N, t=1 | Zona S, t=1 | Zona N, t=2 | Zona S, t=2 |
|:---:|:---:|:---:|:---:|:---:|
| CDR 1 | 5 | 8 | 6 | 9 |
| CDR 2 | 7 | 4 | 8 | 5 |
| CDR 3 | 6 | 6 | 5 | 7 |

Demandas $d_{zt}$ (ton):

| Zona \ Período | $t=1$ (invierno) | $t=2$ (verano) |
|:---:|:---:|:---:|
| Zona N | 30 | 25 |
| Zona S | 20 | 35 |

<a id="ex1-a"></a>

### Parte (a) — Formulación Matemática

Defina claramente **conjuntos, parámetros, variables de decisión, función objetivo y restricciones**. La formulación debe ser **completamente algebraica** — use símbolos de parámetros, no valores numéricos.

Incluya los parámetros $K_{\min}$ (mínimo de CDRs a abrir) y $F_{\max}$ (presupuesto máximo de costos fijos) en la lista de parámetros, y úselos en las restricciones correspondientes.

Identifique además:
- Cuáles variables son binarias y por qué.
- Cuáles restricciones usan la técnica Big-M y cómo funciona el parámetro $u_k$ como constante M natural.
- Cómo se modela la restricción lógica "CDR 1 implica CDR 3" con una sola desigualdad.

> 📝 **Respuesta:**
>
> **Conjuntos:**
>
> **Parámetros:**
>
> **Variables de decisión:**
>
> **Función objetivo:**
>
> **Restricciones:**

<a id="ex1-b"></a>

### Parte (b) — Implementación en Gurobipy

In [6]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

# ─── Conjuntos ───────────────────────────────────────────────────────────────
cdrs   = [1, 2, 3]
zonas  = ['N', 'S']
periodos = [1, 2]

# ─── Parámetros ──────────────────────────────────────────────────────────────
costo_fijo = {1: 10, 2: 12, 3: 8}
capacidad  = {1: 50, 2: 60, 3: 40}

costo_var = {
    (1, 'N', 1): 5,  (1, 'S', 1): 8,  (1, 'N', 2): 6,  (1, 'S', 2): 9,
    (2, 'N', 1): 7,  (2, 'S', 1): 4,  (2, 'N', 2): 8,  (2, 'S', 2): 5,
    (3, 'N', 1): 6,  (3, 'S', 1): 6,  (3, 'N', 2): 5,  (3, 'S', 2): 7,
}

demanda = {
    ('N', 1): 30, ('S', 1): 20,
    ('N', 2): 25, ('S', 2): 35,
}

presupuesto_fijo = 28

#### CÓDIGO AQUÍ ####
modelo = gp.Model("Distribución de Farmacos.")

y = modelo.addVars(cdrs, vtype = GRB.BINARY, name = "Se abre el cdr")
x = modelo.addVars(cdrs, zonas, periodos, vtype = GRB.CONTINUOUS, lb = 0, name = "tons")

cf = gp.quicksum(y[k]*costo_fijo[k] for k in cdrs)
cv = gp.quicksum(x[k,z,t]*costo_var[k,z,t] for k in cdrs for z in zonas for t in periodos)

modelo.setObjective(cf+cv, GRB.MINIMIZE)

modelo.addConstrs(gp.quicksum(x[k,z,t] for k in cdrs)>= demanda[z, t] for z in zonas for t in periodos)
modelo.addConstr(gp.quicksum(y[k] for k in cdrs)>= 2)
modelo.addConstr(y[1]<= y[3])
modelo.addConstr(gp.quicksum(y[k]*costo_fijo[k] for k in cdrs) <= presupuesto_fijo)
modelo.addConstrs(x[k,z,t]<= y[k]*capacidad[k] for k in cdrs for z in zonas for t in periodos)


modelo.optimize()
#####################

if modelo.status == GRB.OPTIMAL:
    print("\n" + "="*55)
    print(f"  SOLUCIÓN ÓPTIMA — Costo Total: {modelo.ObjVal:.2f} MM$")
    print("="*55)

    print("\nCDRs abiertos:")
    for k in cdrs:
        estado = "ABIERTO" if y[k].X > 0.5 else "cerrado"
        print(f"  CDR {k}: {estado}  (costo fijo = {costo_fijo[k]} MM$)")

    print("\nPlan de despacho (ton):")
    filas = []
    for k in cdrs:
        for z in zonas:
            for t in periodos:
                val = x[k, z, t].X
                if val > 1e-6:
                    filas.append({'CDR': k, 'Zona': z, 'Período': t, 'Despacho (ton)': round(val, 2)})
    df = pd.DataFrame(filas)
    print(df.to_string(index=False))

    costo_f = sum(costo_fijo[k] * y[k].X for k in cdrs)
    costo_v = sum(costo_var[k, z, t] * x[k, z, t].X
                  for k in cdrs for z in zonas for t in periodos)
    print(f"\nDesglose de costos:")
    print(f"  Costos fijos:    {costo_f:.2f} MM$")
    print(f"  Costos variables:{costo_v:.2f} MM$  (en $/ton escalados)")
    print(f"  Total:           {modelo.ObjVal:.2f} MM$")

Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-11800H @ 2.30GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 19 rows, 15 columns and 44 nonzeros
Model fingerprint: 0x3039ca52
Variable types: 12 continuous, 3 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [4e+00, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e+00, 4e+01]
Presolve removed 19 rows and 15 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 16 available processors)

Solution count 1: 580 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.800000000000e+02, best bound 5.800000000000e+02, gap 0.0000%

  SOLUCIÓN ÓPTIMA — Costo Total: 580.00 MM$

CDRs abiertos:
  CDR 1: 

<a id="ex1-c"></a>

### Parte (c) — Análisis e Interpretación

Una vez obtenida la solución óptima, responda las siguientes preguntas.

**1. ¿Qué CDRs se abren en la solución óptima y por qué es consistente con las restricciones lógicas?**

> 📝 **Respuesta:**

**2. ¿Qué restricción R2 (capacidad condicional) está activa en el óptimo y qué implicancia tiene?**

> 📝 **Respuesta:**

**3. Si se relajara la restricción lógica R4 (CDR 1 ya no requiere CDR 3), ¿podría mejorar el costo total? Justifique razonando sobre las características del problema sin necesidad de resolver un nuevo modelo.**

> 📝 **Respuesta:**

<a id="ex2"></a>

---

## Ejercicio 2 — LP Integral: Simplex Tableau, Dual y Análisis de Sensibilidad

### Contexto de Negocio: Viñedos del Valle S.A.

**Viñedos del Valle S.A.** es una empresa vitivinícola que elabora dos tipos de vino: **Reserva** ($x_1$, en miles de botellas) y **Gran Reserva** ($x_2$, en miles de botellas). El proceso productivo utiliza tres recursos compartidos:

| Recurso | Reserva ($x_1$) | Gran Reserva ($x_2$) | Disponible |
|:---|:---:|:---:|:---:|
| Uvas seleccionadas (toneladas) | $1$ | $1$ | $\leq 6$ ton |
| Horas de barrica (hrs) | $1$ | $2$ | $\leq 10$ hrs |
| Etiquetado artesanal (hrs) | $1$ | $0$ | $\leq 4$ hrs |
| **Margen neto (M\$/miles bot.)** | **\$2** | **\$3** | — |

La empresa desea **maximizar el margen neto** semanal.

### Formulación

$$\max \quad z = 2x_1 + 3x_2$$

$$\text{s.a.} \quad \begin{cases}
x_1 + x_2 \leq 6 & \text{(uvas)} \\
x_1 + 2x_2 \leq 10 & \text{(barrica)} \\
x_1 \leq 4 & \text{(etiquetado)} \\
x_1,\, x_2 \geq 0
\end{cases}$$

**Forma estándar** (variables de holgura $s_1, s_2, s_3 \geq 0$):

$$\max \quad z = 2x_1 + 3x_2 \quad \text{s.a.} \quad x_1 + x_2 + s_1 = 6, \quad x_1 + 2x_2 + s_2 = 10, \quad x_1 + s_3 = 4$$

<a id="ex2-a"></a>

### Parte (a) — Forma Estándar y Tableau Inicial

El sistema extendido $[A \mid I]$ con columnas para $x_1, x_2, s_1, s_2, s_3$ es:

$$[A \mid I] = \begin{bmatrix} 1 & 1 & 1 & 0 & 0 \\ 1 & 2 & 0 & 1 & 0 \\ 1 & 0 & 0 & 0 & 1 \end{bmatrix}$$

**Base inicial:** $\{s_1, s_2, s_3\}$ — vértice $(0, 0)$, $z = 0$.

**Complete el Tableau Inicial (Iteración 0):**

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | | | | | | |
| $s_1$ | | | | | | |
| $s_2$ | | | | | | |
| $s_3$ | | | | | | |

¿Qué variable entra? __________ ¿Qué variable sale? __________ ¿Cuál es el pivote? __________

<a id="ex2-b"></a>

### Parte (b) — Iteración 1

Muestre explícitamente las operaciones de fila y complete el tableau.

**Operaciones de fila:**

$$R_{\text{nueva pivote}} = \underline{\hspace{8cm}}$$

$$R_{\text{CR}}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{s_1}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{s_3}^{\text{nueva}} = \underline{\hspace{8cm}}$$

**Tableau — Iteración 1:**

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |

¿Es óptimo? _____ ¿Qué variable entra ahora? __________ ¿Cuál sale? __________ ¿Cuál es el pivote? __________

#### Iteración 2 — Solución Óptima

**Operaciones de fila:**

$$R_{\text{nueva pivote}} = \underline{\hspace{8cm}}$$

$$R_{\text{CR}}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{x_2}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{s_3}^{\text{nueva}} = \underline{\hspace{8cm}}$$

**Tableau Óptimo — Iteración 2:**

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |

$$\boxed{x_1^* = \underline{\quad}, \quad x_2^* = \underline{\quad}, \quad z^* = \underline{\quad} \text{ M\$}}$$

### Iteración 2 — Solución Óptima

**Operaciones de fila:**

$$R_{\text{nueva pivote}} = \underline{\hspace{8cm}}$$

$$R_{\text{CR}}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{x_2}^{\text{nueva}} = \underline{\hspace{8cm}}$$

$$R_{s_3}^{\text{nueva}} = \underline{\hspace{8cm}}$$

**Tableau Óptimo — Iteración 2:**

| Base | $x_1$ | $x_2$ | $s_1$ | $s_2$ | $s_3$ | RHS |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **CR** | | | | | | |
| | | | | | | |
| | | | | | | |
| | | | | | | |

**Condición de optimalidad:** ¿Se cumple? _____ ¿Por qué? _______________________________

$$\boxed{x_1^* = \underline{\quad}, \quad x_2^* = \underline{\quad}, \quad z^* = \underline{\quad} \text{ M\$}}$$

Interprete los valores de $s_1$, $s_2$ y $s_3$ en el contexto de Viñedos del Valle:

- $s_1 = \_\_$: _______________________________________________
- $s_2 = \_\_$: _______________________________________________
- $s_3 = \_\_$: _______________________________________________

<a id="ex2-c"></a>

### Parte (c) — Formulación del Dual y Precios Sombra

**Plantee el problema dual** del primal de Viñedos del Valle:

$$\underline{\text{(objetivo)}} \quad w = \underline{\hspace{6cm}}$$

$$\text{s.a.} \quad \begin{cases}
\underline{\hspace{6cm}} & \text{(restricción dual para } x_1\text{)} \\
\underline{\hspace{6cm}} & \text{(restricción dual para } x_2\text{)} \\
y_1, y_2, y_3 \geq 0
\end{cases}$$

**Identifique los precios sombra** a partir del tableau óptimo (fila CR, columnas de las holguras):

$$y_1^* = \underline{\quad}, \qquad y_2^* = \underline{\quad}, \qquad y_3^* = \underline{\quad}$$

**Dualidad fuerte:** $w^* = \underline{\hspace{4cm}} = \underline{\quad} \stackrel{?}{=} z^* = \underline{\quad}$

> 📝 **Interpretación económica:**

<a id="ex2-d"></a>

### Parte (d) — Análisis de Sensibilidad sobre $b_1$ (uvas)

La base óptima es $B = \{x_1, x_2, s_3\}$. Dados:

$$B = \begin{bmatrix} 1 & 1 & 0 \\ 1 & 2 & 0 \\ 1 & 0 & 1 \end{bmatrix}, \qquad B^{-1} = \begin{bmatrix} 2 & -1 & 0 \\ -1 & 1 & 0 \\ -2 & 1 & 1 \end{bmatrix}$$

Sea $b_1 \to 6 + \Delta$. La primera columna de $B^{-1}$ es $d_1 = [\underline{\;},\underline{\;},\underline{\;}]^\top$.

$$x_B = \begin{bmatrix}\underline{\quad}\\\underline{\quad}\\\underline{\quad}\end{bmatrix} + \Delta \begin{bmatrix}\underline{\quad}\\\underline{\quad}\\\underline{\quad}\end{bmatrix} \geq 0 \quad\Rightarrow\quad \Delta \geq \underline{\quad} \;\text{ y }\; \Delta \leq \underline{\quad}$$

$$\boxed{\underline{\quad} \leq b_1 \leq \underline{\quad} \text{ toneladas}}$$

> 📝 **Interpretación:**

<a id="ex2-e"></a>

### Parte (e) — Análisis de Sensibilidad sobre $c_1$ (margen del Reserva)

$x_1$ es básica. Con $c_1 \to 2 + \Delta$, calcule los nuevos costos reducidos de $s_1$ y $s_2$:

$$\bar{c}_{s_1} = [2+\Delta,\; 3,\; 0] \cdot \begin{bmatrix}2\\-1\\-2\end{bmatrix} = \underline{\hspace{4cm}} \geq 0 \;\Rightarrow\; \Delta \geq \underline{\quad}$$

$$\bar{c}_{s_2} = [2+\Delta,\; 3,\; 0] \cdot \begin{bmatrix}-1\\1\\1\end{bmatrix} = \underline{\hspace{4cm}} \geq 0 \;\Rightarrow\; \Delta \leq \underline{\quad}$$

$$\boxed{\underline{\quad} \leq c_1 \leq \underline{\quad} \text{ M\$/miles bot.}}$$

> 📝 **Interpretación:**

<a id="ex2-f"></a>

### Parte (f) — Verificación Computacional

Implemente el problema con `scipy.optimize.linprog` **y** `gurobipy`, reporte la solución óptima, los precios sombra y los rangos de sensibilidad. Compare con los resultados manuales.

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Datos del problema
# max z = 2x1 + 3x2  →  min -2x1 - 3x2

#### CÓDIGO AQUÍ ####

#####################

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# Implementar con Gurobi y reportar:
# - Solucion optima (x1*, x2*, z*)
# - Precios sombra (constr.Pi)
# - Rangos de sensibilidad (var.SAObjLow/Up, constr.SARHSLow/Up)

#### CÓDIGO AQUÍ ####

#####################

### Tabla Comparativa de Resultados

Complete la tabla con los valores obtenidos:

| Método | $x_1^*$ | $x_2^*$ | $z^*$ | $y_1^*$ | $y_2^*$ | $y_3^*$ |
|:---|:---:|:---:|:---:|:---:|:---:|:---:|
| Simplex Tableau (manual) | | | | | | |
| `scipy.optimize.linprog` | | | | | | |
| `gurobipy` | | | | | | |

**Rangos de sensibilidad:**

| Parámetro | Rango manual | Rango Gurobi |
|:---|:---:|:---:|
| $b_1$ (uvas, ton) | | |
| $c_1$ (margen Reserva) | | |

<a id="ex3"></a>

---

## Ejercicio 3 — Dualidad, Holguras Complementarias y KKT

### Contexto de Negocio: Operadora Turística Costera

Una **operadora turística costera** programa excursiones en dos modalidades: **Kayak** ($x_1$) y **Vela** ($x_2$), en decenas de grupos por semana.

$$\max \quad z = x_1 + 2x_2 \qquad \text{s.a.} \quad \begin{cases} x_1 + x_2 \leq 5 \\ x_1 + 3x_2 \leq 9 \\ x_1,\, x_2 \geq 0 \end{cases}$$

> **Dato proporcionado:** La solución óptima del dual es $y_1^* = \dfrac{1}{2}$ y $y_2^* = \dfrac{1}{2}$.

<a id="ex3-a"></a>

### Parte (a) — Recuperar $x^*$ mediante Holguras Complementarias

Use las condiciones de holgura complementaria para recuperar $x_1^*$ y $x_2^*$ a partir de $y^* = (1/2,\; 1/2)$, **sin resolver el primal directamente**.

Como $y_1^* = \underline{\quad} > 0$ → restricción 1 **activa**: $\underline{\hspace{4cm}} = \underline{\quad}$ … (1)

Como $y_2^* = \underline{\quad} > 0$ → restricción 2 **activa**: $\underline{\hspace{4cm}} = \underline{\quad}$ … (2)

Resuelva el sistema (1)–(2):

$$\underline{\hspace{8cm}}$$

$$\boxed{x_1^* = \underline{\quad}, \quad x_2^* = \underline{\quad}, \quad z^* = \underline{\quad} \text{ M\$}}$$

> 📝 **Respuesta:**

<a id="ex3-b"></a>

### Parte (b) — Verificación de Condiciones KKT

Verifique que el par $(x^*, y^*)$ satisface las cuatro condiciones KKT. Recuerde: $\mu_i = y_i^*$.

**1. Estacionaridad** ($A^\top y^* = c$):

$$\begin{bmatrix}\underline{\quad}&\underline{\quad}\\\underline{\quad}&\underline{\quad}\end{bmatrix}\begin{bmatrix}1/2\\1/2\end{bmatrix} = \begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix} \stackrel{?}{=} \begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix}$$  ✓ / ✗

**2. Holgura complementaria** ($y_i^*(b_i - a_i^\top x^*) = 0$):

- R1: $\underline{\quad}(\underline{\quad} - \underline{\quad}) = \underline{\quad}$ ✓ / ✗
- R2: $\underline{\quad}(\underline{\quad} - \underline{\quad}) = \underline{\quad}$ ✓ / ✗

**3. Factibilidad primal:** _(verifique las restricciones)_

> 📝 **Respuesta:**

**4. Factibilidad dual:** $y_1^*, y_2^* \geq 0$ ✓ / ✗

**Conexión conceptual:** ¿Por qué $\mu_i \equiv y_i^*$? ¿Qué implica esto para la relación entre KKT y dualidad?

> 📝 **Respuesta:**

<a id="ex3-c"></a>

### Parte (c) — Análisis de Sensibilidad sobre $b_2$ (embarcaciones)

Dados $B = \begin{bmatrix}1&1\\1&3\end{bmatrix}$ y $B^{-1} = \begin{bmatrix}3/2&-1/2\\-1/2&1/2\end{bmatrix}$.

Sea $b_2 \to 9 + \Delta$. Segunda columna de $B^{-1}$: $d_2 = [\underline{\;},\underline{\;}]^\top$.

$$x_B = \begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix} + \Delta\begin{bmatrix}\underline{\quad}\\\underline{\quad}\end{bmatrix} \geq 0 \quad\Rightarrow\quad \Delta \leq \underline{\quad} \;\text{ y }\; \Delta \geq \underline{\quad}$$

$$\boxed{\underline{\quad} \leq b_2 \leq \underline{\quad}}$$

¿En cuánto cambia $z^*$ si se agregan 4 horas de embarcaciones?

$$\Delta z^* = y_2^* \cdot \Delta b_2 = \underline{\quad} \cdot \underline{\quad} = \underline{\quad} \text{ M\$}$$

> 📝 **Respuesta:**

<a id="ex3-d"></a>

### Parte (d) — Verificación con scipy

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Resolver el Ejercicio 3 con scipy y verificar:
# 1. x1*, x2*, z*
# 2. y1*, y2* (precios sombra / variables duales)
# 3. Condiciones KKT (estacionaridad, HC, dualidad fuerte)

#### CÓDIGO AQUÍ ####

#####################

<a id="resumen"></a>

---

## Resumen de la Ayudantía

### Conexiones entre bloques del curso

| Bloque | Concepto central | Conexión con otro bloque |
|:---|:---|:---|
| **1 — Optimización NL** | KKT: $\mu_i g_i(x^*)=0$, $\nabla f = \sum \mu_i \nabla g_i$ | $\mu_i \equiv y_i^*$ del Bloque 4 |
| **2 — Modelamiento MIP** | Big-M: $x \leq M y$ vincula continua con binaria | Las holguras del LP base son las variables duales del Bloque 4 |
| **3 — Simplex** | $B^{-1}$ genera precios sombra en la fila CR | $c_B B^{-1} = y^{\top}$ (precios sombra = variables duales óptimas) |
| **4 — Dualidad** | $z^* = w^*$, $y_i^* = \partial z^*/\partial b_i$ | KKT (Bloque 1) y Holguras Complementarias son equivalentes |

### Tabla de herramientas computacionales

| Tarea | Herramienta | Función/Método clave |
|:---|:---|:---|
| Optimización no lineal sin restricciones | `scipy.optimize.minimize` | `method='BFGS'` |
| Optimización no lineal con restricciones | `scipy.optimize.minimize` | `method='SLSQP'` |
| LP (verificación) | `scipy.optimize.linprog` | `method='highs'`; `.ineqlin.marginals` |
| LP / MIP (industrial) | `gurobipy` | `.optimize()`, `.Pi`, `.SAObjLow/Up`, `.SARHSLow/Up` |
| Operaciones matriciales Simplex | `numpy` | `np.linalg.inv(B)`, `c_B @ B_inv` |

---

*Ayudantía 9 (Final) — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T1*